# Hillstrom Email Marketing — Causal Uplift Modeling

**Business problem:** Which customers should receive a promotional email **because the email is expected to increase their purchase probability?**

Follow the exact 9-stage workflow. We compare S-learner vs T-learner for **Men’s email vs No E-mail** and **Women’s email vs No E-mail**, separately. This is appropriate for the three-arm randomized Hillstrom study; we do not mistake one email treatment for the other.

**Caution:** Uplift is an estimated conditional average treatment effect, not proof of an individual customer's counterfactual outcome. This notebook must be executed to obtain actual scores, not assumed values.

## 1. Import Libraries
Install and import tools for classification, randomized uplift evaluation, plots, and deployment.

In [ ]:
%pip -q install scikit-uplift scikit-learn pandas numpy scipy matplotlib joblib streamlit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklift.datasets import fetch_hillstrom
from sklift.metrics import qini_auc_score, uplift_auc_score, uplift_at_k
from sklift.viz import plot_qini_curve, plot_uplift_curve
SEED=42

## 2. Data Collection
The Hillstrom dataset records randomized assignment to **No E-Mail**, **Mens E-Mail**, and **Womens E-Mail**. We use `conversion` as the binary purchase outcome.

In [ ]:
bundle=fetch_hillstrom(target_col="all")
df=bundle.data.copy().reset_index(drop=True)
df["segment"]=pd.Series(bundle.treatment).reset_index(drop=True)
for name in bundle.target.columns:
    df[name]=np.asarray(bundle.target[name])
print("Records, columns:",df.shape)
display(df.head())
display(df.groupby('segment').agg(customers=('conversion','size'), conversion_rate=('conversion','mean'), average_spend=('spend','mean')))
print('Group labels:',df.segment.unique())

## 3. Data Preparation
Exclude `visit` and `spend` from predictors, because they are **post-treatment outcomes**. Also exclude `conversion`, the target, and `segment`, the treatment indicator, from the baseline customer attributes. Randomized assignment allows comparing treatment vs control for causal uplift.

In [ ]:
assert {'segment','conversion','visit','spend'}.issubset(df.columns)
control_label=next((x for x in df.segment.unique() if 'no e-mail' in str(x).lower() or 'no email' in str(x).lower()),None)
assert control_label is not None, 'Inspect experiment labels and set control_label manually.'
treatment_labels=[s for s in df.segment.unique() if s!=control_label]
assert len(treatment_labels)==2
predictors=df.drop(columns=['segment','conversion','visit','spend']).columns.tolist()
assert set(predictors).isdisjoint({'conversion','visit','spend','segment'})
print('Control:',control_label,'Treatments:',treatment_labels)
print('Pretreatment features:',predictors)
for label in treatment_labels:
    sub=df[df.segment.isin([control_label,label])]
    print(label,sub.shape,sub.groupby('segment').conversion.mean().to_dict())

## 4. Data Cleaning (if needed)
Inspect missingness and data types. **Do not drop identical records** just because their attributes match: there is no unique customer ID establishing duplicate customers. Missing-value imputation is fitted only on the training partition.

In [ ]:
display(df.isna().sum().to_frame('missing_values'))
display(df.dtypes.astype(str).to_frame('dtype'))
print('Identical rows (not deleted):',df.duplicated().sum())
assert df.conversion.dropna().isin([0,1]).all()
assert not df['conversion'].isna().any()
num_cols=df[predictors].select_dtypes(include=np.number).columns.tolist()
cat_cols=[col for col in predictors if col not in num_cols]
print('Numeric:',num_cols,'Categorical:',cat_cols)

## 5. Model Building
**S-learner:** fit one model with `email_received` as a feature and score each customer twice (1 vs 0). **T-learner:** fit separate treated and untreated classifiers, then subtract predicted probabilities. Both are causal *estimation strategies* applied to randomized data. Random Forest captures nonlinear interactions. Compare them by **Qini AUC** on held-out randomized data, not classification accuracy.

In [ ]:
preprocess=ColumnTransformer([
 ('numeric',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_cols),
 ('categorical',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat_cols)
])
base_classifier=RandomForestClassifier(n_estimators=140,min_samples_leaf=20,random_state=SEED,n_jobs=-1)

def make_pipeline(features):
    nums=[c for c in features if c in num_cols or c=='email_received']
    cats=[c for c in features if c not in nums]
    transformer=ColumnTransformer([
      ('numeric',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),nums),
      ('categorical',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cats)
    ])
    return Pipeline([('preprocess',transformer),('model',clone(base_classifier))])

def fit_s_learner(X,y,t):
    model=make_pipeline(predictors+['email_received'])
    X1=X.copy();X1['email_received']=np.asarray(t)
    model.fit(X1,y)
    return model

def predict_s_uplift(model,X):
    treated=X.copy();treated['email_received']=1
    control=X.copy();control['email_received']=0
    return model.predict_proba(treated)[:,1]-model.predict_proba(control)[:,1]

def fit_t_learner(X,y,t):
    t=np.asarray(t).astype(bool)
    treated_model=make_pipeline(predictors)
    control_model=make_pipeline(predictors)
    treated_model.fit(X.loc[t],y.loc[t]);control_model.fit(X.loc[~t],y.loc[~t])
    return treated_model,control_model

def predict_t_uplift(models,X):
    return models[0].predict_proba(X)[:,1]-models[1].predict_proba(X)[:,1]
print('S-learner and T-learner defined.')

## 6. Model Training
For each email campaign, split experiment participants into **60% train / 20% validation / 20% test**, stratifying by treatment and conversion together where possible. The final test set is untouched until model selection is complete.

In [ ]:
experiments={}
for label in treatment_labels:
    exp=df.loc[df.segment.isin([control_label,label])].copy().reset_index(drop=True)
    exp['email_received']=(exp.segment==label).astype(int)
    strat=exp.email_received.astype(str)+'_'+exp.conversion.astype(str)
    train_idx,remaining_idx=train_test_split(np.arange(len(exp)),test_size=.4,random_state=SEED,stratify=strat)
    remaining=exp.iloc[remaining_idx]
    remaining_strat=remaining.email_received.astype(str)+'_'+remaining.conversion.astype(str)
    valid_idx,test_idx=train_test_split(remaining_idx,test_size=.5,random_state=SEED,stratify=remaining_strat)
    train=exp.iloc[train_idx];valid=exp.iloc[valid_idx];test=exp.iloc[test_idx]
    Xtr=train[predictors];ytr=train.conversion.astype(int);ttr=train.email_received.astype(int)
    s=fit_s_learner(Xtr,ytr,ttr)
    t=fit_t_learner(Xtr,ytr,ttr)
    experiments[str(label)]={'train':train,'valid':valid,'test':test,'S-learner':s,'T-learner':t}
    print(label,'trained:',len(train),'validation:',len(valid),'test:',len(test))

## 7. Model Testing
First select the best approach **on validation data only**, then test that selected approach once. Qini-AUC and uplift-AUC measure how well the ranking prioritizes customers who appear to benefit incrementally. Positive Qini scores are desirable; inspect curves rather than relying on a single value.

In [ ]:
validation_rows=[]
for label,run in experiments.items():
    val=run['valid'];Xv=val[predictors];yv=val.conversion.astype(int).to_numpy();tv=val.email_received.to_numpy()
    for name in ['S-learner','T-learner']:
        preds=predict_s_uplift(run[name],Xv) if name=='S-learner' else predict_t_uplift(run[name],Xv)
        validation_rows.append({'campaign':label,'model':name,'val_qini_auc':qini_auc_score(yv,preds,tv),'val_uplift_auc':uplift_auc_score(yv,preds,tv)})
validation_results=pd.DataFrame(validation_rows).sort_values(['campaign','val_qini_auc'],ascending=[True,False])
display(validation_results)
for label,run in experiments.items():
    selected=validation_results.loc[validation_results.campaign==label].iloc[0]['model']
    run['selected_model_name']=selected
print('Model selections locked prior to evaluating the test set')

In [ ]:
test_rows=[]
for label,run in experiments.items():
    test=run['test'];Xt=test[predictors];yt=test.conversion.astype(int).to_numpy();tt=test.email_received.to_numpy()
    chosen=run['selected_model_name']
    preds=predict_s_uplift(run[chosen],Xt) if chosen=='S-learner' else predict_t_uplift(run[chosen],Xt)
    run['test_scores']=preds
    test_rows.append({'campaign':label,'chosen_model':chosen,'test_qini_auc':qini_auc_score(yt,preds,tt),
                      'test_uplift_auc':uplift_auc_score(yt,preds,tt),
                      'uplift_at_top_30pct':uplift_at_k(yt,preds,tt,strategy='overall',k=.3)})
    plot_qini_curve(yt,preds,tt);plt.title(f'Qini — {label} ({chosen})');plt.show()
    plot_uplift_curve(yt,preds,tt);plt.title(f'Uplift curve — {label}');plt.show()
test_results=pd.DataFrame(test_rows)
display(test_results)
print('Top 30% uplift is an estimate from randomized test groups, not guaranteed profit.')

## 8. Model Validation
**Sanity checks:** confirm randomized group sizes and observed conversion effects. Compare test uplift in the top-ranked 30% against the overall randomized effect, while recognizing small subsets have high uncertainty. Do not retrain or choose a model based on these test outcomes.

In [ ]:
audit=[]
for label,run in experiments.items():
    test=run['test'].copy();test['uplift_score']=run['test_scores']
    top=test.nlargest(max(1,int(round(len(test)*.30))), 'uplift_score')
    for title,frame in [('All test customers',test),('Top 30% ranked',top)]:
        rates=frame.groupby('email_received').conversion.agg(['mean','size'])
        if len(rates)==2:
            audit.append({'campaign':label,'population':title,'treated_count':int(rates.loc[1,'size']),
              'control_count':int(rates.loc[0,'size']),
              'treated_rate':float(rates.loc[1,'mean']),'control_rate':float(rates.loc[0,'mean']),
              'observed_uplift_pp':100*float(rates.loc[1,'mean']-rates.loc[0,'mean'])})
display(pd.DataFrame(audit))
print('Model results require prospective randomized campaign validation before production use.')

## 9. Deployment
Save the **validation-selected** S-/T-learner models and feature schema for each campaign. The Streamlit app lets you enter customer attributes and see estimated incremental conversion for each campaign. **Do not interpret this as a guaranteed personal outcome or automatically send mail.**

In [ ]:
export={'predictors':predictors,'campaigns':{}}
for label,run in experiments.items():
    export['campaigns'][label]={'strategy':run['selected_model_name'],'model':run[run['selected_model_name']]}
joblib.dump(export,'hillstrom_uplift_models.joblib')
print('Saved hillstrom_uplift_models.joblib (load only files you trust).')

In [ ]:
app_text = r'''import streamlit as st
import pandas as pd
import numpy as np
import joblib

st.set_page_config(page_title="Email Uplift Predictor")
st.title("Email Campaign Uplift Predictor")
st.caption("Estimated incremental conversion probability; no email is the baseline. Not a guarantee.")
bundle=joblib.load("hillstrom_uplift_models.joblib")
features=bundle["predictors"]
# Human-editable fields and safe defaults based on Hillstrom schema.
values={}
for column in features:
    if column == "recency": values[column]=st.number_input(column,min_value=0,value=5,step=1)
    elif column == "history": values[column]=st.number_input(column,min_value=0.0,value=100.0)
    elif column == "mens": values[column]=st.selectbox(column,[0,1])
    elif column == "womens": values[column]=st.selectbox(column,[0,1])
    elif column == "zip_code": values[column]=st.selectbox(column,["Urban","Suburban","Rural"])
    elif column == "newbie": values[column]=st.selectbox(column,[0,1])
    elif column == "channel": values[column]=st.selectbox(column,["Phone","Web","Multichannel"])
    else: values[column]=st.text_input(column,"Unknown")

if st.button("Estimate campaign uplift"):
    X=pd.DataFrame([values],columns=features)
    scores=[]
    for label,details in bundle['campaigns'].items():
        if details['strategy']=='S-learner':
            a=X.assign(email_received=1);b=X.assign(email_received=0)
            uplift=details['model'].predict_proba(a)[:,1][0]-details['model'].predict_proba(b)[:,1][0]
        else:
            tr,ctrl=details['model']
            uplift=tr.predict_proba(X)[:,1][0]-ctrl.predict_proba(X)[:,1][0]
        scores.append({'Campaign':label,'Predicted uplift (percentage points)':round(float(uplift)*100,2)})
    st.dataframe(pd.DataFrame(scores),hide_index=True)
    st.info("Negative uplift suggests sending that campaign may not help. Validate decisions via randomized trials and account for email costs.")
'''
Path('app.py').write_text(app_text,encoding='utf-8')
Path('requirements.txt').write_text('streamlit\npandas\nnumpy\nscikit-learn\njoblib\n',encoding='utf-8')
print('Created app.py and requirements.txt')

### How to run the deployment
1. Run all notebook cells in Google Colab.
2. Download `hillstrom_uplift_models.joblib`, `app.py`, and `requirements.txt` from Colab's Files panel.
3. Place all three together in a GitHub repository and deploy via [Streamlit Community Cloud](https://share.streamlit.io/), with main file `app.py`.
4. Verify the app inputs against your training schema. Record library versions and pin them for deployment compatibility.

**Project interpretation:** Ordinary propensity models target likely buyers; uplift models prioritize *incremental* purchases caused by sending an email. Always check incremental business value, campaign costs, confidence intervals, and future experiments before rolling out decisions.